# Colab 08 — ¿Puedo determinar el valor de g en este laboratorio?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 8 — 30/09

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/08_Determinacion_de_g_y_comparacion_de_modelos.ipynb)

Es la medición canónica del laboratorio de mecánica y tiene una particularidad: el valor verdadero se conoce con muchísima precisión, así que no hay dónde esconderse.

**Al terminar vas a poder:** determinar $g$ con su incerteza, comparar dos modelos anidados, y —el punto central— distinguir la incerteza del ajuste de la incerteza del resultado.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Caída libre

$$x(t) = x_0 + v_0 t + \tfrac{1}{2} g t^2$$

El modelo no es lineal en la variable pero **sí es lineal en los
parámetros**, así que cuadrados mínimos anda perfecto y sin semillas.

In [ ]:
generador = np.random.default_rng(23)

# Datos de ejemplo: caída de aproximadamente un metro, registrada en 14
# posiciones. Hay un rozamiento que NO está en el modelo. Ese es el punto.
t = np.linspace(0.06, 0.45, 14)
g_real, b_real = 9.797, 1.5
x_verdadero = 0.5*g_real*t**2 - (b_real/6)*t**3
sigma_x = 0.0006
x = x_verdadero + generador.normal(0, sigma_x, size=len(t))
sx = np.full(len(t), sigma_x)

fig, ax = plt.subplots()
ax.errorbar(t, x, yerr=sx, fmt="o", capsize=3)
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Distancia recorrida (m)")
plt.show()

In [ ]:
def caida(t, x0, v0, g):
    return x0 + v0*t + 0.5*g*t**2


p, e, cov = lab.ajustar(caida, t, x, yerr=sx,
                        nombres=["x0 (m)", "v0 (m/s)", "g (m/s²)"])

g_medido, sg_medido = p[2], e[2]
lab.reportar(g_medido, sg_medido, "m/s²", nombre="g")

### 2. Comparar con el valor local

En Buenos Aires $g = 9{,}797$ m/s² (varía con la latitud y la altura; el
valor tabulado tiene una incerteza despreciable frente a la nuestra).

In [ ]:
g_local = 9.797

z = lab.compatibilidad(g_medido, sg_medido, g_local, 0.0005,
                       etiquetas=("mi medición", "valor local"))

### 3. El punto central de la clase

Si te dio a varios sigma, la reacción instintiva es "voy a medir más veces".
**No sirve.**

`np.sqrt(np.diag(pcov))` devuelve la incerteza **estadística** del
parámetro, y viene con una condición que casi nunca se enuncia: *suponiendo
que el modelo es correcto y que las barras de error son correctas*. No
contiene ningún error sistemático, porque el ajuste no tiene manera de saber
que existe.

Cuando el ajuste dice $g = 9{,}43 \pm 0{,}03$ m/s², está diciendo: *dentro de
mi modelo, éste es el número*. La discrepancia con el valor local vive
**afuera** del modelo. Medir cien veces más divide la barra estadística por
diez y deja la discrepancia intacta: lo único que lográs es pasar de estar
mal a 15σ a estar mal a 150σ.

Los sospechosos concretos en este experimento: rozamiento con el aire, masa
efectiva de la polea, error de alineación del riel, error de cero en la
posición inicial. Todos empujan $g$ **hacia abajo**, que es la firma
característica.

### 4. Buscarlo en los residuos

In [ ]:
c2r, pval = lab.chi2_reducido(x, caida(t, *p), sx, 3)

fig, axes = lab.grafico_con_residuos(
    t, x, caida, p, yerr=sx, normalizar_residuos=True,
    xlabel="Tiempo (s)", ylabel="Distancia (m)",
    titulo=f"Modelo sin rozamiento — χ²_ν = {c2r:.2f}, p = {pval:.2g}")
plt.show()

Mirá bien los dos diagnósticos, porque dicen cosas distintas.

El $\chi^2_\nu$ da alrededor de 2, con un p-valor de un par de por ciento:
sospechoso, pero de esos que uno deja pasar si viene apurado. Los **residuos**,
en cambio, no dejan lugar a dudas: tienen una curvatura clarísima, que es lo
que el modelo cuadrático no puede reproducir.

Y acá está la lección más incómoda de la clase: con ese $\chi^2_\nu$ apenas
elevado, el resultado está a **quince desviaciones estándar** del valor
correcto. Un ajuste puede ser casi aceptable estadísticamente y estar
catastróficamente mal. Por eso el panel de residuos no es opcional.

### 5. Modelos anidados

Agregamos un término de rozamiento a primer orden. El modelo nuevo
**contiene** al anterior (se recupera con $b = 0$): eso es lo que significa
que estén anidados, y permite compararlos limpiamente.

In [ ]:
def caida_con_roce(t, x0, v0, g, b):
    return x0 + v0*t + 0.5*g*t**2 - (b/6)*t**3


p2, e2, cov2 = lab.ajustar(caida_con_roce, t, x, yerr=sx,
                           p0=[0, 0, 9.8, 0.1],
                           nombres=["x0 (m)", "v0 (m/s)", "g (m/s²)", "b"])

c2r2, pval2 = lab.chi2_reducido(x, caida_con_roce(t, *p2), sx, 4, verbose=False)

print()
print(f"modelo simple    : χ²_ν = {c2r:6.2f}   p = {pval:.2e}   "
      f"g = {lab.formatear(p[2], e[2])}")
print(f"con rozamiento   : χ²_ν = {c2r2:6.2f}   p = {pval2:.3f}   "
      f"g = {lab.formatear(p2[2], e2[2])}")

In [ ]:
fig, axes = lab.grafico_con_residuos(
    t, x, caida_con_roce, p2, yerr=sx, normalizar_residuos=True,
    xlabel="Tiempo (s)", ylabel="Distancia (m)",
    titulo=f"Con rozamiento — χ²_ν = {c2r2:.2f}, p = {pval2:.3f}")
plt.show()

lab.compatibilidad(p2[2], e2[2], g_local, 0.0005,
                   etiquetas=("g con rozamiento", "valor local"))

Con el término extra los residuos se vuelven ruido, el $\chi^2_\nu$ se
normaliza y $g$ pasa a ser compatible con el valor local. Ésa es la secuencia
completa del oficio: el ajuste da un número, el diagnóstico dice que falta
algo, los residuos dicen qué, el modelo nuevo lo incorpora y el resultado se
corrige.

**Pero fijate el precio.** La incerteza de $g$ se multiplicó por siete. La
razón está en la matriz de correlación: $g$ y $b$ salen con $\rho \approx
0{,}99$, o sea que el ajuste casi no puede distinguir entre "más gravedad y
más rozamiento" y "menos de las dos". Con estos datos no se pueden determinar
las dos cosas a la vez con buena precisión.

Ésa es una conclusión legítima y hay que escribirla tal cual: *el modelo
completo describe bien los datos, pero el rango medido no permite separar $g$
del rozamiento; para hacerlo habría que medir en un rango de tiempo más
amplio, donde el término cúbico se despegue del cuadrático.*

**Cuidado con la trampa opuesta.** Siempre se puede agregar parámetros hasta
que los residuos se aplanen; con $N$ parámetros y $N$ puntos el ajuste es
perfecto y no significa nada. Un parámetro extra se justifica si (i) tiene
sentido físico, (ii) mejora el $\chi^2_\nu$ de manera sustancial, y (iii)
sale **distinto de cero** con su propia incerteza. Verificá ese último punto
con el valor de $b$ que te dio.

### 6. Nunca uses R² para esto

In [ ]:
print(f"R² del modelo simple  : {lab.R2(x, caida(t, *p)):.6f}")
print(f"R² con rozamiento     : {lab.R2(x, caida_con_roce(t, *p2)):.6f}")
print()
print("Los dos R² son indistinguibles. El χ²_ν se lleva un factor grande.")
print("R² no sirve para elegir entre modelos, y menos si no son lineales.")

### 7. Combinar dos determinaciones independientes

Si además determinaste $g$ por plano inclinado —extrapolando la aceleración
a $\theta = 90°$— y las dos son compatibles, se combinan pesando por
$1/\sigma^2$ (Clase 3).

In [ ]:
g_inclinado, sg_inclinado = 9.781, 0.021

z = lab.compatibilidad(p2[2], e2[2], g_inclinado, sg_inclinado,
                       etiquetas=("caída libre", "plano inclinado"))

if z < 3:
    print()
    print("Compatibles: tiene sentido combinarlas.")
    lab.promedio_ponderado([p2[2], g_inclinado], [e2[2], sg_inclinado])

Mirá quién manda en esa combinación: el resultado final es casi exactamente
el del plano inclinado. Es la consecuencia cuadrática que vimos en la Clase 3
—la medición más precisa domina— y acá tiene una lectura práctica: si el
método de caída libre no puede separar $g$ del rozamiento, sumarlo aporta
poco. Combinar mediciones no es un trámite de cierre: a veces el resultado
honesto es que una de las dos no agrega información.

### 8. Cómo se informa el resultado final

Con las dos incertezas **separadas y con la fuente identificada**:

> $g = 9{,}79 \pm 0{,}02_{\text{(est)}} \pm 0{,}05_{\text{(sist)}}$ m/s²,
> donde la sistemática corresponde a la incerteza en la calibración de la
> escala de posición.

La estadística sale del ajuste. La sistemática **la estimás vos**, y estimarla
es parte del trabajo, no un agregado opcional. La forma habitual es variar el
factor sospechoso dentro de su rango razonable y ver cuánto se mueve el
resultado.

### 9. Ejercicios

1. Determiná $g$ con tus datos por los dos métodos y hacé el análisis
   completo: compatibilidad, búsqueda del sistemático, combinación.
2. Estimá la sistemática por calibración: si tu escala de posición tuviera un
   error de escala del 0,5 %, ¿cuánto cambia $g$? (Multiplicá todas las
   posiciones por 1,005 y rehacé el ajuste.)
3. Ajustá con un modelo de rozamiento cuadrático en lugar de lineal.
   ¿Podés distinguir cuál es el correcto con estos datos? Si no podés,
   decilo: es una conclusión legítima.
4. ¿Qué pasa con la incerteza de $g$ si medís en un rango de tiempo el doble
   de largo? Simulá esa situación agregando puntos y explicá por qué la
   mejora es más grande que $1/\sqrt{N}$.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Informe 3 (Clases 7 y 8)

Determinación de $g$ por dos métodos independientes, con análisis de
compatibilidad y una sección explícita de errores sistemáticos.